# Imports

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import einsum, rearrange, reduce, repeat

# Set Device

In [3]:
device = 'cpu'
if torch.cuda.is_available():
    device='cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device='mps'

print("Device set:", device)

Device set: mps


# Load Data

In [4]:
!pwd

/Users/irabandutta/Developer/2026-08-llm-from-scratch/notebooks


In [5]:
train_bin_file_path = '../data/tinystories/processed/train.bin'
tokens = np.fromfile(train_bin_file_path, dtype=np.uint16)

In [6]:
print(len(tokens))
print('-'*100)
print(tokens[:100])

471872517
----------------------------------------------------------------------------------------------------
[ 3198  1110    11   257  1310  2576  3706 20037  1043   257 17598   287
   607  2119    13  1375  2993   340   373  2408   284   711   351   340
   780   340   373  7786    13 20037  2227   284  2648   262 17598   351
   607  1995    11   523   673   714 34249   257  4936   319   607 10147
    13   198   198    43   813  1816   284   607  1995   290   531    11
   366 29252    11   314  1043   428 17598    13  1680   345  2648   340
   351   502   290 34249   616 10147  1701  2332  1995 13541   290   531
    11   366  5297    11 20037    11   356   460  2648   262 17598   290
  4259   534 10147   526]


In [7]:
# Unique tokens in TinyStories
np.unique_counts(tokens)

UniqueCountsResult(values=array([    0,     1,     2, ..., 50243, 50244, 50255],
      shape=(29251,), dtype=uint16), counts=array([1857907, 1573680,      11, ...,     105,       8,    4296],
      shape=(29251,)))

# Data Split: Train, Val, Test

In [8]:
token_limit = 10000
train_length = int(0.8*token_limit)
val_len = int(0.1*token_limit)

tokens_subset = tokens[:token_limit]
train_tokens = torch.tensor(tokens_subset[:train_length])
val_tokens = torch.tensor(tokens_subset[train_length:(train_length+val_len)])
test_tokens = torch.tensor(tokens_subset[(train_length+val_len):])

print('#Tokens-Train:', len(train_tokens))
print('#Tokens-Val:', len(val_tokens))
print('#Tokens-Test:', len(test_tokens))

#Tokens-Train: 8000
#Tokens-Val: 1000
#Tokens-Test: 1000


In [9]:
# Create a smaple batch from train data
# Batch of train samples: We want to view this as a B, T tensor
# B: Batch size & T: Sequence length

B, T = 4, 8
buffer = train_tokens[:(B*T)+1]
X = buffer[:-1].view(B,-1)
y = buffer[1:].view(B,-1)

print('X:\n', X)
print('-'*100)
print('y:\n', y)

X:
 tensor([[ 3198,  1110,    11,   257,  1310,  2576,  3706, 20037],
        [ 1043,   257, 17598,   287,   607,  2119,    13,  1375],
        [ 2993,   340,   373,  2408,   284,   711,   351,   340],
        [  780,   340,   373,  7786,    13, 20037,  2227,   284]],
       dtype=torch.uint16)
----------------------------------------------------------------------------------------------------
y:
 tensor([[ 1110,    11,   257,  1310,  2576,  3706, 20037,  1043],
        [  257, 17598,   287,   607,  2119,    13,  1375,  2993],
        [  340,   373,  2408,   284,   711,   351,   340,   780],
        [  340,   373,  7786,    13, 20037,  2227,   284,  2648]],
       dtype=torch.uint16)


In [10]:
import tiktoken

enc = tiktoken.get_encoding('gpt2')
enc.n_vocab

50257

# Einops & Einsum Quick Reference

```python
from einops import rearrange, reduce, repeat
from einops import einsum
```

---

# What is einops?

`einops` is a library for expressing tensor operations using the **meaning of dimensions** instead of axis numbers.

Instead of writing things like

```python
x = x.view(B, T, H, Dh).transpose(1, 2)
```

you can write

```python
x = rearrange(x, "b t (h d) -> b h t d", h=H)
```

The operation itself becomes documentation.

---

# The 4 operations you'll use most

- `rearrange` → reshape, transpose, flatten, split dimensions
- `repeat` → duplicate data along dimensions
- `reduce` → sum/mean/max while optionally reshaping
- `einsum` → matrix multiplication, dot products, tensor contractions

---

# 1. rearrange

Think of `rearrange` as replacing:

- `reshape`
- `view`
- `permute`
- `transpose`
- `flatten`
- combinations of the above

---

## Example 1 — Transpose

Suppose

```python
x.shape = (2, 3, 4)      # (batch, height, width)
```

Want

```python
(2, 4, 3)
```

```python
y = rearrange(x, "b h w -> b w h")
```

### Common DL use case

Changing between

```text
(B, C, H, W)
```

and

```text
(B, H, W, C)
```

---

## Example 2 — Flatten dimensions

Suppose

```python
x.shape = (2, 3, 4)
```

Want

```python
(2, 12)
```

```python
y = rearrange(x, "b h w -> b (h w)")
```

### Common DL use case

Flatten CNN feature maps before a linear layer.

---

## Example 3 — Split a dimension

Suppose

```python
x.shape = (2, 12)
```

Want

```python
(2, 3, 4)
```

```python
y = rearrange(x, "b (h w) -> b h w", h=3)
```

### Common DL use case

Splitting embedding dimension into multiple attention heads.

---

## Example 4 — Multi-head Attention ⭐⭐⭐⭐⭐

Suppose

```python
Q.shape = (B, T, D)

B = Batch size
T = Sequence length
D = Embedding dimension
```

Assume

```python
D = H × Dh
```

Need

```python
(B, H, T, Dh)
```

```python
Q = rearrange(Q, "b t (h d) -> b h t d", h=H)
```

Later, combine heads again

```python
Q = rearrange(Q, "b h t d -> b t (h d)")
```

### Common DL use case

Every Transformer implementation.

---

# 2. repeat

`repeat` **duplicates data along dimensions**.

Unlike `rearrange`, which only changes the organization of existing elements, `repeat` actually creates multiple copies of the data.

---

## Example 1 — Replicate across attention heads

Suppose

    x.shape = (5, 128)    # (sequence, embedding)

Want one copy for each of 8 attention heads:

    y = repeat(x, "n d -> h n d", h=8)

Result:

    (8, 5, 128)

Each head receives an identical copy of `x`.

**### Common DL use case**

Replicating a tensor across attention heads.

---

## Example 2 — GQA: Replicate KV heads ⭐⭐⭐⭐⭐

Suppose:

    K.shape = (B, Hkv, T, D)

with:

    Hq  = 4
    Hkv = 2

In GQA, every KV head is shared by 2 query heads:

    Query heads:  0   1   2   3
    KV heads:     0   0   1   1

So we need to replicate each KV head twice:

    K = repeat(
        K,
        "b g t d -> b (g r) t d",
        r=2
    )

Before:

    K.shape = (B, 2, T, D)

After:

    K.shape = (B, 4, T, D)

The mapping is:

    KV head 0 → K head 0, K head 1
    KV head 1 → K head 2, K head 3

Now standard attention can be computed:

    scores = einsum(
        Q,
        K,
        "b h i d, b h j d -> b h i j"
    )

### Why is `repeat` needed?

The original GQA tensors have:

    Q: (B, 4, T, D)
    K: (B, 2, T, D)

Here:

    h = query head
    g = KV head

`einsum` does **not** automatically infer the GQA mapping:

    Q0 → K0
    Q1 → K0
    Q2 → K1
    Q3 → K1

Instead, we explicitly create the mapping with `repeat`:

    (B, 2, T, D)
          ↓ repeat
    (B, 4, T, D)

Then `einsum` operates on the matching `h` dimension.

---

## Example 3 — Repeat elements within a dimension

Suppose:

    x = torch.tensor([1, 2, 3])

We want:

    [1, 1, 2, 2, 3, 3]

We can write:

    y = repeat(x, "n -> (n r)", r=2)

Result:

    [1, 1, 2, 2, 3, 3]

**### Common DL use case**

Repeating labels, indices, or tokens when constructing structured batches.

---

### `repeat` vs `rearrange`

A useful distinction:

    rearrange → reorganize existing elements
    repeat    → duplicate elements

For example:

    rearrange(x, "b h d -> b d h")

only changes the arrangement.

Whereas:

    repeat(x, "b h d -> b (h r) d", r=2)

actually duplicates every head.

---

# 3. reduce

Performs reductions while describing dimensions.

---

## Example

Suppose

```python
x.shape = (2, 3, 4)
```

Average over width.

```python
y = reduce(x, "b h w -> b h", "mean")
```

Equivalent to

```python
x.mean(dim=2)
```

### Common DL use case

Global average pooling.

---

# 4. einsum

`einsum` is **NOT** another rearranging function.

Instead, it performs **mathematical operations** like

- dot products
- matrix multiplication
- batched matrix multiplication
- attention
- tensor contractions

whereas

`rearrange` only changes how data is viewed.

A good mental model is

| Function | Does math? |
|-----------|------------|
| rearrange | ❌ |
| repeat | ❌ |
| reduce | ✅ (reduces values) |
| einsum | ✅ (multiplies & sums) |

---

# Understanding einsum

The notation labels dimensions using letters.

Example

```text
b = batch
t = sequence
d = embedding
h = heads
```

When a letter appears in **both inputs but disappears from the output**, that dimension is summed over.

---

## Example 1 — Dot Product

Vectors

```python
a.shape = (4,)
b.shape = (4,)
```

```python
y = einsum(a, b, "d, d ->")
```

Equivalent to

```python
(a * b).sum()
```

### Common DL use case

Computing similarity between embeddings.

---

## Example 2 — Matrix Multiplication ⭐⭐⭐⭐⭐

Suppose

```python
A.shape = (3, 4)
B.shape = (4, 5)
```

Want

```python
(3, 5)
```

```python
C = einsum(A, B, "m k, k n -> m n")
```

Equivalent to

```python
A @ B
```

Notice

```
k
```

appears in both inputs but not the output.

Therefore it is summed over.

---

## Example 3 — Batched Matrix Multiplication

Suppose

```python
A.shape = (B, M, K)
B.shape = (B, K, N)
```

```python
C = einsum(A, B, "b m k, b k n -> b m n")
```

Equivalent to

```python
A @ B
```

for every batch independently.

### Common DL use case

Batched linear algebra.

---

## Example 4 — Computing Attention Scores ⭐⭐⭐⭐⭐

Suppose

```python
Q.shape = (B, H, T, D)
K.shape = (B, H, T, D)
```

Need

```python
scores.shape = (B, H, T, T)
```

```python
scores = einsum(
    Q,
    K,
    "b h i d, b h j d -> b h i j"
)
```

Notice

```
d
```

disappears.

Therefore

```
Σ_d
```

is performed automatically.

This computes

```
Q · Kᵀ
```

without explicitly calling `.transpose()`.

### Common DL use case

Transformer attention.

---

## Example 5 — Attention Output ⭐⭐⭐⭐⭐

Suppose

```python
scores.shape = (B, H, T, T)
V.shape      = (B, H, T, D)
```

Need

```python
(B, H, T, D)
```

```python
output = einsum(
    scores,
    V,
    "b h i j, b h j d -> b h i d"
)
```

Equivalent to

```python
scores @ V
```

### Common DL use case

Final attention output in every Transformer.

---

## Example 6 — Linear Layer

Suppose

```python
X.shape = (B, T, D)
W.shape = (D, O)
```

Need

```python
(B, T, O)
```

```python
Y = einsum(
    X,
    W,
    "b t d, d o -> b t o"
)
```

Equivalent to

```python
X @ W
```

### Common DL use case

Fully connected layers.

---

# Rearrange vs Einsum

Suppose

```python
Q.shape = (B, T, D)
```

Split into heads

```python
Q = rearrange(Q, "b t (h d) -> b h t d", h=H)
```

Compute attention

```python
scores = einsum(
    Q,
    K,
    "b h i d, b h j d -> b h i j"
)
```

These solve **completely different problems**.

- `rearrange` changes **where** data lives.
- `einsum` computes **new values**.

---

# Rule for reading einsum

Whenever you see

```text
a b c,
a c d
      ↓
a b d
```

Ask yourself:

> Which dimension disappeared?

Here

```
c
```

disappeared.

Therefore

```
Σc
```

was computed.

That's the only rule you need to read almost every `einsum` expression you'll encounter in deep learning.

---

# Cheat Sheet

| Operation | Typical DL Use |
|------------|----------------|
| `rearrange` | Split/combine attention heads, flatten CNN features, patchify images |
| `repeat` | Duplicate embeddings, positional encodings, head replication |
| `reduce` | Global average pooling, mean/sum across dimensions |
| `einsum` | Matrix multiplication, batched matmul, attention, linear layers, tensor contractions |

# MHA - Rough

In [11]:
# Gentle Digression
x = torch.randint(4, (2, 4, 2))
print(x)
print(x.view(2, -1))
torch.cat(torch.unbind(x, 1), 1)

tensor([[[0, 3],
         [3, 3],
         [3, 0],
         [2, 1]],

        [[2, 1],
         [3, 2],
         [1, 3],
         [0, 0]]])
tensor([[0, 3, 3, 3, 3, 0, 2, 1],
        [2, 1, 3, 2, 1, 3, 0, 0]])


tensor([[0, 3, 3, 3, 3, 0, 2, 1],
        [2, 1, 3, 2, 1, 3, 0, 0]])

Start Attention

In [12]:
# Linear Projection
a = torch.randn(2, 20, 32)
l = nn.Linear(32, 96)
x = l(a)
print(a.shape)
print(x.shape)

torch.Size([2, 20, 32])
torch.Size([2, 20, 96])


In [13]:
# Combined q,k,v projection
s = torch.randn(2, 20, 3*32)
s1 = s[:, :, 0*32:1*32]
s2 = s[:, :, 1*32:2*32]
s3 = s[:, :, 2*32:3*32]

s1_sp, s2_sp, s3_sp = torch.split(s, 32, dim=-1)

assert (s1==s1_sp).any()
assert (s2==s2_sp).any()
assert (s3==s3_sp).any()

In [14]:
# Reshaping q, k, v due to n_heads
n_heads = 4
q = torch.randn(2, 20, 32)

q1 = q.view(2, 20, n_heads, -1)
q1 = q1.transpose(1, 2)
print(q1.shape)

q2 = rearrange(q, "b s (h d) -> b h s d", h=n_heads)
print(q2.shape)

torch.Size([2, 4, 20, 8])
torch.Size([2, 4, 20, 8])


In [15]:
# Testing einsum
torch.manual_seed(42)
q = torch.randn(2, 4, 20, 8)
k = torch.randn(2, 4, 20, 8)

attn_scores1 = q @ k.transpose(2, 3)
attn_scores2 = einsum(q, k, "b h i d, b h j d -> b h i j")

print(attn_scores1.shape)
print(attn_scores2.shape)

try:
    assert (attn_scores1==attn_scores2).all()
    assert (attn_scores1==torch.rand(2, 4, 20, 20)).all()
    # print(1/0)
except AssertionError as e:
    print('AssertionError')
except Exception as e:
    print(e)

torch.Size([2, 4, 20, 20])
torch.Size([2, 4, 20, 20])
AssertionError


In [16]:
# Masking
torch.manual_seed(42)
mask = torch.ones(5, 5)
print(mask)
print('-'*50)
for i in range(len(mask)):
    mask[i, (i+1):] = -torch.inf
print(mask)
print('-'*50)
A = torch.randn(5, 5)
A *= mask
print(A)
print('-'*50)
print('This is wrong since we (-ve) attn scores get multipled with -inf to create +inf')

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])
--------------------------------------------------
tensor([[1., -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1.]])
--------------------------------------------------
tensor([[ 1.9269,    -inf,    -inf,     inf,    -inf],
        [-1.2345, -0.0431,     inf,     inf,     inf],
        [-0.4934,  0.2415, -1.1109,    -inf,     inf],
        [-0.2168, -1.3847, -0.3957,  0.8034,     inf],
        [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]])
--------------------------------------------------
This is wrong since we (-ve) attn scores get multipled with -inf to create +inf


In [17]:
# Mask by Addition
torch.manual_seed(42)
m1 = torch.full((5, 5), fill_value=-torch.inf)
print(m1)
print('-'*50)
m1 = torch.triu(m1)
print(m1)
print('-'*50)
m1 = torch.triu(m1, diagonal=1)
print(m1)
print('-'*50)
A = torch.randn(5, 5)
print(A)
print('-'*50)
A += m1
print(A)
print('-'*50)
print('This Causal Mask by Addition')


tensor([[-inf, -inf, -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf, -inf]])
--------------------------------------------------
tensor([[-inf, -inf, -inf, -inf, -inf],
        [0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf]])
--------------------------------------------------
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])
--------------------------------------------------
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
        [-1.2345, -0.0431, -1.6047, -0.7521, -0.6866],
        [-0.4934,  0.2415, -1.1109,  0.0915, -2.3169],
        [-0.2168, -1.3847, -0.3957,  0.8034, -0.6216],
        [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]])
------------------

In [18]:
# Mask by masked_fill()
torch.manual_seed(42)
m2 = torch.ones(5,5)
print(m2)
print('-'*50)
m2 = torch.tril(m2)==0
print(m2)
print('-'*50)
A = torch.randn(5, 5)
print(A)
print('-'*50)
A = A.masked_fill(m2, value=-torch.inf)
print(A)
print('-'*50)
print('This is Causal Mask by masked_fill')


tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])
--------------------------------------------------
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
--------------------------------------------------
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
        [-1.2345, -0.0431, -1.6047, -0.7521, -0.6866],
        [-0.4934,  0.2415, -1.1109,  0.0915, -2.3169],
        [-0.2168, -1.3847, -0.3957,  0.8034, -0.6216],
        [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]])
--------------------------------------------------
tensor([[ 1.9269,    -inf,    -inf,    -inf,    -inf],
        [-1.2345, -0.0431,    -inf,    -inf,    -inf],
        [-0.4934,  0.2415, -1.1109,    -inf,    -inf],
        [-0.2168, -1.3847, -0

In [19]:
torch.manual_seed(42)
A = torch.randn(2, 2, 5, 5)
print(A)
print('-'*50)
print(m2)
print('-'*50)
A = A.masked_fill(m2, value=-torch.inf)
print(A)
print('-'*50)

tensor([[[[ 1.9269e+00,  1.4873e+00,  9.0072e-01, -2.1055e+00,  6.7842e-01],
          [-1.2345e+00, -4.3067e-02, -1.6047e+00, -7.5214e-01,  1.6487e+00],
          [-3.9248e-01, -1.4036e+00, -7.2788e-01, -5.5943e-01, -7.6884e-01],
          [ 7.6245e-01,  1.6423e+00, -1.5960e-01, -4.9740e-01,  4.3959e-01],
          [-7.5813e-01,  1.0783e+00,  8.0080e-01,  1.6806e+00,  1.2791e+00]],

         [[ 1.2964e+00,  6.1047e-01,  1.3347e+00, -2.3162e-01,  4.1759e-02],
          [-2.5158e-01,  8.5986e-01, -1.3847e+00, -8.7124e-01, -2.2337e-01],
          [ 1.7174e+00,  3.1888e-01, -4.2452e-01,  3.0572e-01, -7.7459e-01],
          [-1.5576e+00,  9.9564e-01, -8.7979e-01, -6.0114e-01, -1.2742e+00],
          [ 2.1228e+00, -1.2347e+00, -4.8791e-01, -9.1382e-01, -6.5814e-01]]],


        [[[ 7.8024e-02,  5.2581e-01, -4.8799e-01,  1.1914e+00, -8.1401e-01],
          [-7.3599e-01, -1.4032e+00,  3.6004e-02, -6.3477e-02,  6.7561e-01],
          [-9.7807e-02,  1.8446e+00, -1.1845e+00,  1.3835e+00,  1.4451

In [20]:
torch.stack([m2, m2], dim=0)

tensor([[[False,  True,  True,  True,  True],
         [False, False,  True,  True,  True],
         [False, False, False,  True,  True],
         [False, False, False, False,  True],
         [False, False, False, False, False]],

        [[False,  True,  True,  True,  True],
         [False, False,  True,  True,  True],
         [False, False, False,  True,  True],
         [False, False, False, False,  True],
         [False, False, False, False, False]]])

In [21]:
# Check Softmax
F.softmax(attn_scores1, dim=-1).shape

torch.Size([2, 4, 20, 20])

In [22]:
hasattr(F, 'scaled_dot_product_attention')

True

In [23]:
# Sliding Window Attention: From 1st principles

def sliding_window_attention(A:torch.Tensor, window_len:int, debug:bool=False):

    seq_len = A.shape[-1]
    # Capping slide_window to seq_len//2 on higher side
    slide_window = min(window_len, seq_len//2)

    # Normal Causal Attention mask
    mask = torch.ones(seq_len, seq_len, device=A.device)
    mask = torch.tril(mask)==0
    # Code for Sliding Window Attention Mask
    for i in range(len(mask)):
        start_idx = max(0, i-slide_window+1)
        mask[i, :start_idx] = True

    if debug:
        print('mask:\n', mask)
        print('-'*100)
    A = A.masked_fill(mask, value=-torch.inf)
    return A


torch.manual_seed(42)
A = torch.randn(2, 2, 10, 10)
A_sw = sliding_window_attention(A, 3, True)
A_sw[0, 0, :, :]

mask:
 tensor([[False,  True,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True,  True],
        [ True, False, False, False,  True,  True,  True,  True,  True,  True],
        [ True,  True, False, False, False,  True,  True,  True,  True,  True],
        [ True,  True,  True, False, False, False,  True,  True,  True,  True],
        [ True,  True,  True,  True, False, False, False,  True,  True,  True],
        [ True,  True,  True,  True,  True, False, False, False,  True,  True],
        [ True,  True,  True,  True,  True,  True, False, False, False,  True],
        [ True,  True,  True,  True,  True,  True,  True, False, False, False]])
----------------------------------------------------------------------------------------------------


tensor([[ 1.9269,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [-0.3925, -1.4036,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [-0.7581,  1.0783,  0.8008,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [   -inf,  0.8599, -1.3847, -0.8712,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [   -inf,    -inf, -0.8798, -0.6011, -1.2742,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [   -inf,    -inf,    -inf,  1.1914, -0.8140, -0.7360,    -inf,    -inf,
            -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,  1.4451,  0.8564,  2.2181,    -inf,
            -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,  0.4263,  0.5750, -0.6417,
            -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  1.1412,  0.0516,
          0.7440,    -inf],
        [   -inf,  

# Normalization - Rough

BatchNorm

In [25]:
# 2D input
x = torch.randn(20, 3)
x.mean(dim=0, keepdim=True).shape

torch.Size([1, 3])

In [26]:
# 3D input
x = torch.randn(2, 10, 6)
print(x.shape)
x = x.transpose(1, 2)
print(x.shape)
x.mean(dim=(0,2), keepdim=True).shape

torch.Size([2, 10, 6])
torch.Size([2, 6, 10])


torch.Size([1, 6, 1])

LayerNorm

In [27]:
# 2D input
x = torch.randn(20, 3)
x.mean(dim=-1, keepdim=True).shape

torch.Size([20, 1])

In [28]:
# 3D input
x = torch.randn(2, 10, 6)
x.mean(dim=-1, keepdim=True).shape

torch.Size([2, 10, 1])

In [29]:
torch.ones(24).var()

tensor(0.)